# 010 — Self-RAG with LangGraph
**Retrieval Series** | Adaptive retrieval with self-grading

Self-RAG makes the LLM decide:
1. **Should I retrieve?** (maybe the answer is in training data)
2. **Are the retrieved docs relevant?** (grade each one)
3. **Is my generated answer grounded?** (detect hallucinations)
4. **Should I try again?** (loop if needed)

LangGraph models this as a state machine with conditional edges.


In [ ]:
import os, warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', message=".*LangChain.*")
warnings.filterwarnings('ignore', message=".*allowed_objects.*")
warnings.filterwarnings('ignore', module="langchain.*")
warnings.filterwarnings('ignore', module="langgraph.*")

from dotenv import load_dotenv
load_dotenv(os.path.join(os.getcwd(), '..', '.env'))
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
assert GROQ_API_KEY, "GROQ_API_KEY not found — check ../.env"
print("✓ API key loaded")


In [ ]:
import os, time
os.environ["ANONYMIZED_TELEMETRY"] = "False"  # silence ChromaDB telemetry

from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0, max_retries=2)
llm_smart = ChatGroq(model="llama-3.3-70b-versatile", temperature=0, max_retries=2)
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    cache_folder=os.path.join(os.getcwd(), '..', 'datasets', 'models'),
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)
print(f"✓ llm       : {llm.model_name}")
print(f"✓ llm_smart : {llm_smart.model_name}")
print(f"✓ embeddings: all-MiniLM-L6-v2 (384-dim, normalized cosine)")


In [ ]:
import os

DS_DIR = os.path.join(os.getcwd(), '..', 'datasets')
os.makedirs(DS_DIR, exist_ok=True)

def load_doc(fname):
    path = os.path.join(DS_DIR, fname)
    if os.path.exists(path):
        return open(path, encoding='utf-8').read()
    raise FileNotFoundError(
        f"Missing: {path}\n"
        "Run  python create_datasets.py  from the project root."
    )

ai_text  = load_doc("ai.txt")
ml_text  = load_doc("machine_learning.txt")
nlp_text = load_doc("nlp.txt")
dl_text  = load_doc("deep_learning.txt")
rag_text = load_doc("rag.txt")
all_text = "\n\n".join([ai_text, ml_text, nlp_text, dl_text, rag_text])

print(f"✓ 5 dataset files loaded")
print(f"  {'File':<20} {'Chars':>8}  {'~Words':>7}")
print(f"  {'-'*40}")
for name, txt in [('ai.txt', ai_text), ('machine_learning.txt', ml_text),
                  ('nlp.txt', nlp_text), ('deep_learning.txt', dl_text),
                  ('rag.txt', rag_text)]:
    print(f"  {name:<20} {len(txt):>8,}  {len(txt.split()):>7,}")
print(f"  {'-'*40}")
print(f"  {'TOTAL':<20} {len(all_text):>8,}  {len(all_text.split()):>7,}")


In [ ]:
# ── Build retriever ───────────────────────────────────────────────────────
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS

splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=50)
docs = [Document(page_content=c, metadata={"source": "wiki"})
        for text in [ai_text, ml_text, dl_text, nlp_text]
        for c in splitter.split_text(text[:15000])]
vectorstore = FAISS.from_documents(docs, embeddings)
retriever   = vectorstore.as_retriever(search_kwargs={"k": 5})
print(f"✓ Retriever over {len(docs)} docs")


In [ ]:
# ── Grader chains ────────────────────────────────────────────────────────
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Relevance grader: grade 'yes' if the doc contains ANY useful information for the question
relevance_prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "You are a retrieval grader. Assess whether a document contains information "
        "useful for answering a question. Be LENIENT — grade 'yes' if the document "
        "is even partially relevant. Reply with only the word 'yes' or 'no'."
    )),
    ("human", "Document:\n{document}\n\nQuestion: {question}\n\nRelevant? (yes/no):"),
])
relevance_grader = relevance_prompt | llm | StrOutputParser()

# Hallucination grader: is answer grounded in retrieved docs?
hallucination_prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "Check if the answer is grounded in the provided context. "
        "Reply 'yes' if the answer is supported, 'no' if it contains unsupported claims. "
        "Reply with only 'yes' or 'no'."
    )),
    ("human", "Context:\n{context}\n\nAnswer: {answer}\n\nIs this answer supported? (yes/no):"),
])
hallucination_grader = hallucination_prompt | llm | StrOutputParser()

# Quick smoke test
test_verdict = relevance_grader.invoke({
    "document": "Machine learning enables systems to learn from data.",
    "question": "What is machine learning?"
})
print(f"✓ Relevance grader ready   (smoke test: {test_verdict.strip()!r})")
print("✓ Hallucination grader ready")


In [ ]:
# ── Self-RAG State & Nodes ────────────────────────────────────────────────
from typing import TypedDict, List
from langgraph.graph import StateGraph, END

class SelfRAGState(TypedDict):
    question:       str
    documents:      List[str]
    generation:     str
    retries:        int
    hallucination:  str    # 'yes' or 'no'

MAX_RETRIES = 2

def retrieve(state: SelfRAGState):
    docs = retriever.invoke(state["question"])
    return {**state, "documents": [d.page_content for d in docs]}

def grade_documents(state: SelfRAGState):
    q    = state["question"]
    kept, verdicts = [], []
    for doc in state["documents"]:
        verdict = relevance_grader.invoke({"document": doc[:500], "question": q})
        is_relevant = "yes" in verdict.lower()
        verdicts.append("✓" if is_relevant else "✗")
        if is_relevant:
            kept.append(doc)
    # Fallback: if all graded irrelevant, keep top 2 anyway (avoid empty context)
    if not kept:
        kept = state["documents"][:2]
        print(f"  Relevance grading: [{' '.join(verdicts)}] → all filtered, using top 2 as fallback")
    else:
        print(f"  Relevance grading: [{' '.join(verdicts)}] → {len(kept)}/{len(state['documents'])} kept")
    return {**state, "documents": kept}

def generate(state: SelfRAGState):
    ctx = "\n\n".join(state["documents"][:3])
    gen_prompt = ChatPromptTemplate.from_template(
        "Answer based only on context.\n\nContext:\n{ctx}\n\nQuestion: {q}\n\nAnswer:"
    )
    ans = (gen_prompt | llm | StrOutputParser()).invoke(
        {"ctx": ctx[:3000], "q": state["question"]}
    )
    return {**state, "generation": ans}

def grade_generation(state: SelfRAGState):
    ctx     = "\n\n".join(state["documents"][:3])
    verdict = hallucination_grader.invoke({
        "context": ctx[:2000],
        "answer":  state["generation"][:800],
    })
    grounded = "yes" in verdict.lower()
    print(f"  Hallucination check: {'✓ grounded' if grounded else '✗ hallucinated'}")
    return {**state, "hallucination": "no" if grounded else "yes",
            "retries": state.get("retries", 0) + 1}

def decide_after_grading(state: SelfRAGState):
    if not state["documents"]:
        return "generate"
    return "generate"

def decide_after_gen(state: SelfRAGState):
    if state["hallucination"] == "no" or state["retries"] >= MAX_RETRIES:
        return "end"
    return "retry"

print("✓ Nodes defined")


In [ ]:
# ── Build LangGraph ──────────────────────────────────────────────────────
graph = StateGraph(SelfRAGState)
graph.add_node("retrieve",        retrieve)
graph.add_node("grade_documents", grade_documents)
graph.add_node("generate",        generate)
graph.add_node("grade_generation",grade_generation)

graph.set_entry_point("retrieve")
graph.add_edge("retrieve",         "grade_documents")
graph.add_conditional_edges("grade_documents", decide_after_grading,
                             {"generate": "generate"})
graph.add_edge("generate",         "grade_generation")
graph.add_conditional_edges("grade_generation", decide_after_gen,
                             {"end": END, "retry": "retrieve"})

self_rag_app = graph.compile()
print("✓ Self-RAG graph compiled")


In [ ]:
# ── Run Self-RAG ──────────────────────────────────────────────────────────
import time

test_questions = [
    "What are the main applications of machine learning in healthcare?",
    "How does backpropagation work in neural networks?",
]

for question in test_questions:
    print(f"\nQ: {question}")
    print("-" * 60)
    t0 = time.time()
    result = self_rag_app.invoke({
        "question":      question,
        "documents":     [],
        "generation":    "",
        "retries":       0,
        "hallucination": "",
    })
    elapsed = time.time() - t0
    print(f"\nAnswer: {result['generation'][:400]}")
    print(f"Retries: {result['retries']}  |  Grounded: {result['hallucination'] == 'no'}  |  Time: {elapsed:.1f}s")


## Key Takeaways
- Self-RAG turns RAG into a **decision-making loop**, not a one-shot call
- The key nodes: retrieve → grade relevance → generate → grade hallucination → loop
- `retries` cap prevents infinite loops on unanswerable questions
- LangGraph's conditional edges make the branching logic explicit and debuggable
